<a href="https://colab.research.google.com/github/gcarboni1/qwen-vl-labs/blob/main/qwen25vl_A_benchmarking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Qwen2.5-VL — Notebook A: Benchmarking & Profiling

**Obiettivo**: misurare con precisione le performance prima di ottimizzare.
Senza baseline non puoi sapere se un'ottimizzazione ha funzionato.

### Metriche che misureremo

| Metrica | Definizione | Rilevante per |
|---|---|---|
| **TTFT** | Time To First Token | Latenza percepita dall'utente |
| **TPS** | Tokens Per Second (generation) | Throughput |
| **E2E Latency** | Tempo totale richiesta→risposta completa | SLA complessivo |
| **VRAM peak** | Memoria GPU massima occupata | Capacity planning |
| **ViT time** | Tempo speso nel visual encoder | Bottleneck analysis |
| **Prefill time** | Tempo per processare i token visivi+testo | Bottleneck analysis |

---

## Setup

In [1]:
# !pip install transformers accelerate qwen-vl-utils torch torchvision
# !pip install matplotlib seaborn pandas psutil gputil
# !pip install flash-attn --no-build-isolation  # opzionale, richiede CUDA

import torch
import time
import gc
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image, ImageDraw
from dataclasses import dataclass, field, asdict
from typing import List, Optional, Dict
from contextlib import contextmanager
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
import warnings
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM totale: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

ModuleNotFoundError: No module named 'qwen_vl_utils'

In [ ]:
# ── Utility: misurazione memoria GPU ──────────────────────────────────────────

def get_vram_usage_gb() -> float:
    """Memoria GPU allocata in GB."""
    if device == "cuda":
        return torch.cuda.memory_allocated() / 1e9
    return 0.0

def get_vram_peak_gb() -> float:
    """Picco massimo di memoria GPU allocata in GB."""
    if device == "cuda":
        return torch.cuda.max_memory_allocated() / 1e9
    return 0.0

def reset_vram_stats():
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()

@contextmanager
def measure_time(label: str, results: dict):
    """Context manager che misura il tempo di un blocco di codice."""
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    yield
    if device == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    results[label] = elapsed
    print(f"  [{label}] {elapsed*1000:.1f} ms")

print("Utility caricate.")

In [ ]:
# ── Caricamento modello ────────────────────────────────────────────────────────

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
# MODEL_ID = "Qwen/Qwen2.5-VL-2B-Instruct"  # se GPU < 16GB

reset_vram_stats()
vram_before_load = get_vram_usage_gb()

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    # attn_implementation="flash_attention_2"  # decommenta se hai flash-attn installato
)
model.eval()

vram_model = get_vram_usage_gb()
print(f"\nVRAM occupata dal modello: {vram_model - vram_before_load:.2f} GB")

---
## Sezione 1 — Profiler del singolo forward pass

Capire dove va il tempo: ViT? Prefill? Generazione?

In [ ]:
def create_test_image(w, h, color=(100, 149, 237)):
    img = Image.new('RGB', (w, h), color=color)
    draw = ImageDraw.Draw(img)
    draw.text((10, 10), f"{w}x{h}", fill=(255,255,255))
    return img

def build_inputs(image, prompt="Descrivi questa immagine in dettaglio."):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt}
        ]
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        padding=True,
        return_tensors="pt"
    ).to(device)
    return inputs


@dataclass
class BenchResult:
    image_size: tuple
    n_visual_tokens: int
    n_input_tokens: int
    n_output_tokens: int
    vit_time_ms: float
    prefill_time_ms: float
    generation_time_ms: float
    e2e_time_ms: float
    ttft_ms: float
    tps: float
    vram_peak_gb: float


def benchmark_single(image, max_new_tokens=128, n_warmup=1, n_runs=3) -> BenchResult:
    """Benchmark completo su una singola immagine."""

    inputs = build_inputs(image)
    _, h, w = inputs['image_grid_thw'][0].tolist()
    n_visual_tokens = h * w
    n_input_tokens = inputs['input_ids'].shape[1]

    print(f"  Immagine: {image.size} | Visual tokens: {n_visual_tokens} | Input tokens: {n_input_tokens}")

    # ── Hook per misurare solo il ViT ─────────────────────────────────────────
    vit_times = []

    def vit_pre_hook(module, input):
        if device == "cuda": torch.cuda.synchronize()
        module._t_start = time.perf_counter()

    def vit_post_hook(module, input, output):
        if device == "cuda": torch.cuda.synchronize()
        vit_times.append((time.perf_counter() - module._t_start) * 1000)

    h1 = model.visual.register_forward_pre_hook(vit_pre_hook)
    h2 = model.visual.register_forward_hook(vit_post_hook)

    # ── Warmup ────────────────────────────────────────────────────────────────
    for _ in range(n_warmup):
        with torch.no_grad():
            _ = model.generate(**inputs, max_new_tokens=16, do_sample=False)
    vit_times.clear()

    # ── Benchmark runs ────────────────────────────────────────────────────────
    e2e_times, prefill_times, gen_times, ttft_list, vram_peaks = [], [], [], [], []
    n_output_tokens = 0

    for run in range(n_runs):
        reset_vram_stats()
        gc.collect()
        if device == "cuda": torch.cuda.empty_cache()

        t_e2e_start = time.perf_counter()

        # Prefill (forward senza generazione)
        if device == "cuda": torch.cuda.synchronize()
        t_prefill_start = time.perf_counter()
        with torch.no_grad():
            prefill_out = model(**inputs, use_cache=True)
        if device == "cuda": torch.cuda.synchronize()
        prefill_times.append((time.perf_counter() - t_prefill_start) * 1000)

        # TTFT: tempo fino al primo token generato
        t_gen_start = time.perf_counter()
        with torch.no_grad():
            first_token = model.generate(
                **inputs, max_new_tokens=1, do_sample=False
            )
        if device == "cuda": torch.cuda.synchronize()
        ttft_list.append((time.perf_counter() - t_e2e_start) * 1000)

        # Generazione completa
        t_gen_full_start = time.perf_counter()
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False
            )
        if device == "cuda": torch.cuda.synchronize()
        t_e2e_end = time.perf_counter()

        n_output_tokens = out.shape[1] - inputs['input_ids'].shape[1]
        gen_times.append((t_e2e_end - t_gen_full_start) * 1000)
        e2e_times.append((t_e2e_end - t_e2e_start) * 1000)
        vram_peaks.append(get_vram_peak_gb())

    h1.remove(); h2.remove()

    avg_vit    = np.mean(vit_times) if vit_times else 0
    avg_pre    = np.mean(prefill_times)
    avg_gen    = np.mean(gen_times)
    avg_e2e    = np.mean(e2e_times)
    avg_ttft   = np.mean(ttft_list)
    avg_vram   = np.mean(vram_peaks)
    tps        = n_output_tokens / (avg_gen / 1000) if avg_gen > 0 else 0

    return BenchResult(
        image_size=image.size,
        n_visual_tokens=n_visual_tokens,
        n_input_tokens=n_input_tokens,
        n_output_tokens=n_output_tokens,
        vit_time_ms=avg_vit,
        prefill_time_ms=avg_pre,
        generation_time_ms=avg_gen,
        e2e_time_ms=avg_e2e,
        ttft_ms=avg_ttft,
        tps=tps,
        vram_peak_gb=avg_vram,
    )

print("Funzioni di benchmark pronte.")

In [ ]:
# Test su una singola immagine per verificare
img_test = create_test_image(512, 512)
print("Benchmark singolo (512×512):")
r = benchmark_single(img_test, max_new_tokens=64, n_warmup=1, n_runs=2)
print(f"\nRisultato:")
print(f"  TTFT:         {r.ttft_ms:.0f} ms")
print(f"  E2E:          {r.e2e_time_ms:.0f} ms")
print(f"  ViT time:     {r.vit_time_ms:.0f} ms ({100*r.vit_time_ms/r.prefill_time_ms:.1f}% del prefill)")
print(f"  TPS:          {r.tps:.1f} tok/s")
print(f"  VRAM peak:    {r.vram_peak_gb:.2f} GB")

---
## Sezione 2 — Impatto della risoluzione sul tempo e sulla VRAM

La risoluzione è il principale knob dell'architettura Qwen2.5-VL.  
Qui vediamo come scala latenza, token visivi e VRAM.

In [ ]:
resolutions = [
    (256,  256),
    (448,  448),
    (672,  672),
    (896,  896),
    (1120, 1120),
    (512,  1024),   # portrait
    (1024, 512),    # landscape
]

results_resolution = []
for w, h in resolutions:
    img = create_test_image(w, h)
    print(f"\nBenchmark {w}×{h}:")
    r = benchmark_single(img, max_new_tokens=64, n_warmup=1, n_runs=2)
    results_resolution.append(r)

df_res = pd.DataFrame([asdict(r) for r in results_resolution])
df_res['label'] = [f"{w}×{h}" for w,h in resolutions]
print("\n\nTabella risultati:")
print(df_res[['label','n_visual_tokens','vit_time_ms','prefill_time_ms','ttft_ms','e2e_time_ms','tps','vram_peak_gb']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Impatto della risoluzione sulle performance — Qwen2.5-VL', fontsize=14)

# Ordina per numero di token visivi
df_plot = df_res.sort_values('n_visual_tokens')
x = df_plot['n_visual_tokens']
labels = df_plot['label']
xi = range(len(df_plot))

def bar_plot(ax, y, title, ylabel, color):
    bars = ax.bar(xi, y, color=color, alpha=0.8, edgecolor='white')
    ax.set_xticks(xi)
    ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=8)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel(ylabel)
    for bar, val in zip(bars, y):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
                f'{val:.0f}', ha='center', va='bottom', fontsize=7)

bar_plot(axes[0,0], df_plot['n_visual_tokens'],  'Token visivi',          'n token',  '#4C9BE8')
bar_plot(axes[0,1], df_plot['vit_time_ms'],      'ViT time',              'ms',       '#E88C4C')
bar_plot(axes[0,2], df_plot['prefill_time_ms'],  'Prefill time',          'ms',       '#E84C6F')
bar_plot(axes[1,0], df_plot['ttft_ms'],          'TTFT',                  'ms',       '#9B4CE8')
bar_plot(axes[1,1], df_plot['tps'],              'Throughput (tok/s)',     'tok/s',    '#4CE87A')
bar_plot(axes[1,2], df_plot['vram_peak_gb'],     'VRAM peak',             'GB',       '#E8D44C')

# Aggiungi annotazione % ViT sul prefill
vit_pct = df_plot['vit_time_ms'] / df_plot['prefill_time_ms'] * 100
ax2 = axes[0,1].twinx()
ax2.plot(xi, vit_pct, 'r--o', markersize=5, label='% sul prefill')
ax2.set_ylabel('% del prefill', color='red')
ax2.legend(loc='upper left', fontsize=7)

plt.tight_layout()
plt.savefig('benchmark_resolution.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nInsight chiave:")
print(f"  ViT come % del prefill: min={vit_pct.min():.0f}%, max={vit_pct.max():.0f}%")
print(f"  → Ad alta risoluzione il ViT diventa il bottleneck principale")

---
## Sezione 3 — Pytorch Profiler: dove va esattamente il tempo?

Il profiler di PyTorch ci permette di vedere le operazioni CUDA kernel per kernel.

In [ ]:
from torch.profiler import profile, record_function, ProfilerActivity

img_profile = create_test_image(448, 448)
inputs_p = build_inputs(img_profile)

# Warmup
with torch.no_grad():
    _ = model(**inputs_p)

activities = [ProfilerActivity.CPU]
if device == "cuda":
    activities.append(ProfilerActivity.CUDA)

with profile(
    activities=activities,
    record_shapes=True,
    with_stack=False,
    profile_memory=True,
) as prof:
    with record_function("full_forward"):
        with torch.no_grad():
            with record_function("vit"):
                # Separiamo manualmente ViT e LLM
                pixel_values = inputs_p['pixel_values']
                grid_thw = inputs_p['image_grid_thw']
                vit_out = model.visual(pixel_values, grid_thw)

            with record_function("full_model"):
                _ = model(**inputs_p)

print("\n=== Top 20 operazioni per tempo CPU ===")
print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=20))

if device == "cuda":
    print("\n=== Top 20 operazioni per tempo CUDA ===")
    print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=20))

In [ ]:
# Breakdown per modulo (ViT vs LLM vs merger)
key_avgs = prof.key_averages()

# Raggruppa per categoria
categories = {
    'vit': 0, 'attention': 0, 'mlp': 0, 'layernorm': 0, 'other': 0
}

for evt in key_avgs:
    name = evt.key.lower()
    t = evt.cpu_time_total / 1e3  # ms
    if 'vit' in name or 'visual' in name:
        categories['vit'] += t
    elif 'attn' in name or 'attention' in name:
        categories['attention'] += t
    elif 'mlp' in name or 'linear' in name:
        categories['mlp'] += t
    elif 'norm' in name:
        categories['layernorm'] += t
    else:
        categories['other'] += t

fig, ax = plt.subplots(figsize=(7, 7))
labels = list(categories.keys())
sizes = list(categories.values())
colors = ['#4C9BE8', '#E84C6F', '#4CE87A', '#E8D44C', '#9B4CE8']

wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors,
    autopct='%1.1f%%', startangle=90,
    pctdistance=0.8
)
ax.set_title('Breakdown tempo CPU per categoria\n(forward pass singolo, 448×448)', fontsize=12)
plt.tight_layout()
plt.savefig('profiler_breakdown.png', dpi=120)
plt.show()

---
## Sezione 4 — Impatto della lunghezza dell'output sul throughput

Il throughput (tok/s) non è costante: dipende dalla lunghezza dell'output  
e dalla pressione della KV cache.

In [ ]:
img_fixed = create_test_image(448, 448)
output_lengths = [16, 32, 64, 128, 256, 512]

results_length = []
for max_tok in output_lengths:
    print(f"max_new_tokens={max_tok}")
    r = benchmark_single(img_fixed, max_new_tokens=max_tok, n_warmup=1, n_runs=2)
    results_length.append({
        'max_new_tokens': max_tok,
        'n_output_tokens': r.n_output_tokens,
        'e2e_ms': r.e2e_time_ms,
        'tps': r.tps,
        'ttft_ms': r.ttft_ms,
        'gen_ms': r.generation_time_ms,
    })

df_len = pd.DataFrame(results_length)
print(df_len.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Throughput al variare della lunghezza output — immagine fissa 448×448', fontsize=12)

axes[0].plot(df_len['n_output_tokens'], df_len['tps'], 'o-', color='#4CE87A', linewidth=2)
axes[0].set_xlabel('Token generati'); axes[0].set_ylabel('tok/s')
axes[0].set_title('Throughput (tok/s)')
axes[0].grid(alpha=0.3)

axes[1].plot(df_len['n_output_tokens'], df_len['e2e_ms'], 'o-', color='#E84C6F', linewidth=2)
axes[1].set_xlabel('Token generati'); axes[1].set_ylabel('ms')
axes[1].set_title('Latenza E2E (ms)')
axes[1].grid(alpha=0.3)

# TTFT è costante (non dipende da max_new_tokens)
axes[2].bar(range(len(df_len)), df_len['ttft_ms'], color='#9B4CE8', alpha=0.8)
axes[2].set_xticks(range(len(df_len)))
axes[2].set_xticklabels(df_len['max_new_tokens'])
axes[2].set_xlabel('max_new_tokens'); axes[2].set_ylabel('ms')
axes[2].set_title('TTFT (dovrebbe essere ~costante)')
axes[2].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('benchmark_output_length.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nInsight: TTFT costante = il prefill (ViT + LLM) domina il primo token.")
print("Se vuoi ridurre TTFT, devi ottimizzare il prefill (risoluzione, Flash Attention).")
print("Se vuoi aumentare il throughput, ottimizza la fase di generazione (quantizzazione, speculative decoding).")

---
## Sezione 5 — Batching: throughput vs latenza

Il trade-off fondamentale in produzione: batch più grandi = più throughput, più latenza.

In [ ]:
def benchmark_batch(batch_size: int, img_size=(448,448), max_new_tokens=64, n_runs=2):
    """Benchmark con batch di immagini."""
    images = [create_test_image(*img_size) for _ in range(batch_size)]

    messages_batch = []
    for img in images:
        messages_batch.append([{
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": "Descrivi brevemente."}
            ]
        }])

    texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
             for m in messages_batch]
    image_inputs_list = [process_vision_info(m)[0] for m in messages_batch]
    all_images = [img for imgs in image_inputs_list for img in imgs]

    inputs = processor(
        text=texts,
        images=all_images,
        padding=True,
        return_tensors="pt"
    ).to(device)

    e2e_times = []
    n_output_tokens = 0

    for _ in range(n_runs):
        reset_vram_stats()
        if device == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        if device == "cuda": torch.cuda.synchronize()
        t1 = time.perf_counter()
        e2e_times.append((t1 - t0) * 1000)
        n_output_tokens = out.shape[1] - inputs['input_ids'].shape[1]

    avg_e2e = np.mean(e2e_times)
    total_tokens = n_output_tokens * batch_size
    throughput = total_tokens / (avg_e2e / 1000)  # tok/s totali
    latency_per_item = avg_e2e / batch_size
    vram = get_vram_peak_gb()

    print(f"  batch={batch_size}: e2e={avg_e2e:.0f}ms | lat/item={latency_per_item:.0f}ms | throughput={throughput:.0f} tok/s | VRAM={vram:.1f}GB")
    return {
        'batch_size': batch_size,
        'e2e_ms': avg_e2e,
        'latency_per_item_ms': latency_per_item,
        'throughput_tps': throughput,
        'vram_gb': vram,
    }

batch_sizes = [1, 2, 4, 8]  # riduci se OOM
results_batch = []
for bs in batch_sizes:
    try:
        r = benchmark_batch(bs)
        results_batch.append(r)
    except RuntimeError as e:
        print(f"  batch={bs}: OOM - {e}")
        break

df_batch = pd.DataFrame(results_batch)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Trade-off Batching: Throughput vs Latenza — Qwen2.5-VL', fontsize=12)

axes[0].bar(df_batch['batch_size'], df_batch['throughput_tps'], color='#4CE87A', alpha=0.8)
axes[0].set_xlabel('Batch size'); axes[0].set_ylabel('tok/s totali')
axes[0].set_title('Throughput totale ↑ con batch')
axes[0].grid(alpha=0.3, axis='y')

axes[1].bar(df_batch['batch_size'], df_batch['latency_per_item_ms'], color='#E84C6F', alpha=0.8)
axes[1].set_xlabel('Batch size'); axes[1].set_ylabel('ms')
axes[1].set_title('Latenza per singola richiesta ↑ con batch')
axes[1].grid(alpha=0.3, axis='y')

axes[2].bar(df_batch['batch_size'], df_batch['vram_gb'], color='#E8D44C', alpha=0.8)
axes[2].set_xlabel('Batch size'); axes[2].set_ylabel('GB')
axes[2].set_title('VRAM cresce con batch')
axes[2].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('benchmark_batching.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nRegola pratica:")
print("  - Applicazione interattiva: batch=1, minimizza latenza")
print("  - Pipeline batch/offline:   batch=4-8, massimizza throughput")
print("  - SLA misto: continuous batching (vLLM) = meglio di entrambi")

---
## Sezione 6 — Baseline report finale

Un report riassuntivo da usare come confronto con le ottimizzazioni (Notebook B).

In [ ]:
# Genera report baseline da salvare come JSON
import platform

baseline = {
    "model_id": MODEL_ID,
    "dtype": "bfloat16",
    "flash_attention": False,  # True se hai usato flash_attention_2
    "hardware": {
        "gpu": torch.cuda.get_device_name(0) if device == "cuda" else "CPU",
        "vram_total_gb": torch.cuda.get_device_properties(0).total_memory / 1e9 if device == "cuda" else 0,
        "python": platform.python_version(),
        "torch": torch.__version__,
    },
    "key_metrics_448x448": {
        "ttft_ms": results_resolution[1].ttft_ms if len(results_resolution) > 1 else None,
        "tps": results_resolution[1].tps if len(results_resolution) > 1 else None,
        "vram_peak_gb": results_resolution[1].vram_peak_gb if len(results_resolution) > 1 else None,
        "n_visual_tokens": results_resolution[1].n_visual_tokens if len(results_resolution) > 1 else None,
    },
    "resolution_scaling": [
        {
            "size": f"{r.image_size[0]}x{r.image_size[1]}",
            "visual_tokens": r.n_visual_tokens,
            "ttft_ms": round(r.ttft_ms, 1),
            "tps": round(r.tps, 1),
        } for r in results_resolution
    ]
}

with open('baseline_report.json', 'w') as f:
    json.dump(baseline, f, indent=2)

print("=== BASELINE REPORT ===")
print(json.dumps(baseline, indent=2))
print("\nSalvato in baseline_report.json")
print("→ Carica questo file nel Notebook B per confrontare le ottimizzazioni")

In [ ]:
# Dashboard visiva finale
fig = plt.figure(figsize=(16, 10))
fig.suptitle(f'Baseline Dashboard — {MODEL_ID} (BF16, no ottimizzazioni)', fontsize=14, fontweight='bold')

df_sorted = df_res.sort_values('n_visual_tokens')
x_labels = df_sorted['label']
xi = range(len(df_sorted))

ax1 = fig.add_subplot(2, 3, 1)
ax1.plot(df_sorted['n_visual_tokens'], df_sorted['ttft_ms'], 'o-', color='#9B4CE8', lw=2)
ax1.set_title('TTFT vs token visivi'); ax1.set_xlabel('Visual tokens'); ax1.set_ylabel('ms'); ax1.grid(alpha=0.3)

ax2 = fig.add_subplot(2, 3, 2)
ax2.plot(df_sorted['n_visual_tokens'], df_sorted['tps'], 'o-', color='#4CE87A', lw=2)
ax2.set_title('Throughput vs token visivi'); ax2.set_xlabel('Visual tokens'); ax2.set_ylabel('tok/s'); ax2.grid(alpha=0.3)

ax3 = fig.add_subplot(2, 3, 3)
ax3.plot(df_sorted['n_visual_tokens'], df_sorted['vram_peak_gb'], 'o-', color='#E8D44C', lw=2)
ax3.set_title('VRAM peak vs token visivi'); ax3.set_xlabel('Visual tokens'); ax3.set_ylabel('GB'); ax3.grid(alpha=0.3)

ax4 = fig.add_subplot(2, 3, 4)
bottom = df_sorted['vit_time_ms'].values
bar1 = ax4.bar(xi, df_sorted['vit_time_ms'], label='ViT', color='#E88C4C', alpha=0.85)
bar2 = ax4.bar(xi, df_sorted['prefill_time_ms'] - bottom, bottom=bottom, label='LLM Prefill', color='#4C9BE8', alpha=0.85)
bar3 = ax4.bar(xi, df_sorted['generation_time_ms'], bottom=df_sorted['prefill_time_ms'], label='Generation', color='#4CE87A', alpha=0.85)
ax4.set_xticks(xi); ax4.set_xticklabels(x_labels, rotation=35, ha='right', fontsize=8)
ax4.set_title('Breakdown tempo per fase'); ax4.set_ylabel('ms'); ax4.legend(fontsize=8)

ax5 = fig.add_subplot(2, 3, 5)
if len(df_batch) > 0:
    ax5_twin = ax5.twinx()
    ax5.bar(df_batch['batch_size'], df_batch['throughput_tps'], color='#4CE87A', alpha=0.7, label='Throughput')
    ax5_twin.plot(df_batch['batch_size'], df_batch['latency_per_item_ms'], 'ro-', label='Latenza/item')
    ax5.set_xlabel('Batch size'); ax5.set_ylabel('tok/s', color='green'); ax5_twin.set_ylabel('ms', color='red')
    ax5.set_title('Throughput vs Latenza per batch')
    lines1, labels1 = ax5.get_legend_handles_labels()
    lines2, labels2 = ax5_twin.get_legend_handles_labels()
    ax5.legend(lines1+lines2, labels1+labels2, fontsize=8)

ax6 = fig.add_subplot(2, 3, 6)
metrics = ['TTFT (ms)', 'TPS', 'VRAM (GB)', 'ViT% prefill']
# Usa i dati della 448x448
r448 = results_resolution[1] if len(results_resolution) > 1 else results_resolution[0]
vit_pct_448 = r448.vit_time_ms / r448.prefill_time_ms * 100
values = [r448.ttft_ms, r448.tps, r448.vram_peak_gb, vit_pct_448]
units  = ['ms', 'tok/s', 'GB', '%']
colors_kpi = ['#9B4CE8', '#4CE87A', '#E8D44C', '#E88C4C']
ax6.axis('off')
for i, (m, v, u, c) in enumerate(zip(metrics, values, units, colors_kpi)):
    ax6.text(0.1, 0.85 - i*0.22, m, transform=ax6.transAxes, fontsize=10, color='gray')
    ax6.text(0.6, 0.85 - i*0.22, f"{v:.1f} {u}", transform=ax6.transAxes, fontsize=14, fontweight='bold', color=c)
ax6.set_title('KPI Baseline (448×448)', fontsize=10)

plt.tight_layout()
plt.savefig('baseline_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()
print("Dashboard salvata. Passa al Notebook B per le ottimizzazioni.")